# Notebook 02: Forward Process 在真实图像上的可视化

**目标**：看 q(x_t | x_0) 在 CIFAR-10/MNIST 真实图像上的效果。直观理解 `linear` 与 `cosine` schedule 的差异。

**前置**：L03 + nb01

In [ ]:
import torch
import math
import matplotlib.pyplot as plt
from torchvision import datasets, transforms

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# 载入 CIFAR-10 一张图（也可换 MNIST）
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize([0.5]*3, [0.5]*3)])
ds = datasets.CIFAR10('./data', train=True, download=True, transform=transform)
x0 = torch.stack([ds[i][0] for i in [0, 7, 42]], dim=0).to(device)  # 3 张图
print(f'x0 shape: {x0.shape}, range: [{x0.min():.2f}, {x0.max():.2f}]')

In [ ]:
T = 1000
def linear_schedule(T):
    betas = torch.linspace(1e-4, 0.02, T)
    return betas, betas.cumprod(0).neg().add(1).cumprod_(0) if False else (1 - betas).cumprod(0)

def cosine_schedule(T, s=0.008):
    t = torch.linspace(0, T, T+1) / T
    f = torch.cos((t + s) / (1+s) * math.pi/2) ** 2
    alpha_bar = f / f[0]
    betas = torch.clip(1 - alpha_bar[1:] / alpha_bar[:-1], 1e-5, 0.999)
    return betas, betas.add(0).neg().add_(1).cumprod(0) if False else (1 - betas).cumprod(0)

betas_lin, ac_lin = linear_schedule(T)
betas_cos, ac_cos = cosine_schedule(T)

plt.plot(ac_lin, label='linear ᾱ_t')
plt.plot(ac_cos, label='cosine ᾱ_t')
plt.xlabel('t'); plt.ylabel('ᾱ_t'); plt.legend(); plt.title('Schedule comparison'); plt.show()

In [ ]:
def q_sample(x0, t, alphas_cumprod):
    eps = torch.randn_like(x0)
    ac = alphas_cumprod[t].view(-1, 1, 1, 1).to(x0.device)
    return ac.sqrt() * x0 + (1 - ac).sqrt() * eps

ts_show = [0, 100, 200, 400, 600, 800, 999]

fig, axes = plt.subplots(2 * x0.shape[0], len(ts_show), figsize=(2*len(ts_show), 2*2*x0.shape[0]))
for row_pair in range(x0.shape[0]):
    for j, t_val in enumerate(ts_show):
        t = torch.tensor([t_val], device=device)
        x_lin = q_sample(x0[row_pair:row_pair+1], t, ac_lin)
        x_cos = q_sample(x0[row_pair:row_pair+1], t, ac_cos)
        for k, x in enumerate([x_lin, x_cos]):
            img = (x[0].cpu() / 2 + 0.5).clamp(0, 1).permute(1, 2, 0).numpy()
            ax = axes[2*row_pair + k][j]
            ax.imshow(img); ax.axis('off')
            if j == 0: ax.set_ylabel(['linear', 'cosine'][k])
            if row_pair == 0 and k == 0: ax.set_title(f't={t_val}')
plt.tight_layout(); plt.show()

## 观察

- Linear：t=200 已经噪声很重，t=400 几乎纯噪声
- Cosine：t=400 还能看到主体轮廓，t=600 才接近纯噪声

Cosine 在中等 t 区域**保留更多有效信号**，给训练更多机会学习。这就是 Improved DDPM 的核心论点。

## 思考题

1. 对低分辨率（如 32×32 CIFAR-10），cosine 的优势真的明显吗？
2. 假设你要做 512×512 高分辨率扩散，应当选哪种 schedule？为什么？
3. 自己实现一个 "S-curve" schedule，让 ᾱ_t 在 t≈500 时刚好为 0.5。试着可视化它的 forward process。